In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_auc_score
import pandas as pd
import os
import joblib

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 0.733145
mimic_train = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_train.csv')
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_val.csv')

In [4]:
train_texts = mimic_train['text'].tolist()
train_labels = mimic_train['label'].tolist()
val_texts = mimic_test['text'].tolist()
val_labels = mimic_test['label'].tolist()

In [5]:
vectorizers = [
    ('tfidf', TfidfVectorizer(lowercase=True)),
    ('count', CountVectorizer(lowercase=True)),
    ('tfidf_sw', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('count_sw', CountVectorizer(lowercase=True, stop_words='english'))
]

In [6]:
results = {}

In [7]:
## --- SVM ---
for vname, vectorizer in vectorizers:
    pipe = Pipeline([
        ('vectorizer', vectorizer),
        ('clf', LinearSVC(class_weight='balanced', max_iter=5000))
    ])
    param_grid = {
        'vectorizer__max_features': [10000, 20000],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'vectorizer__min_df': [3, 5],
        'clf__C': [0.01, 0.1, 1, 10],
    }
    grid = GridSearchCV(pipe, param_grid, cv=3, scoring='f1', verbose=2, n_jobs=-1)
    grid.fit(train_texts, train_labels)
    results[f'SVM_{vname}'] = grid

Fitting 3 folds for each of 32 candidates, totalling 96 fits
Fitting 3 folds for each of 32 candidates, totalling 96 fits
Fitting 3 folds for each of 32 candidates, totalling 96 fits
Fitting 3 folds for each of 32 candidates, totalling 96 fits


In [8]:
## --- Random Forest ---
for vname, vectorizer in vectorizers:
    pipe = Pipeline([
        ('vectorizer', vectorizer),
        ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])
    param_grid = {
        'vectorizer__max_features': [10000, 20000],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'vectorizer__min_df': [3, 5],
        'clf__n_estimators': [100, 300],
        'clf__max_depth': [None, 20, 40],
    }
    grid = GridSearchCV(pipe, param_grid, cv=3, scoring='f1', verbose=2, n_jobs=-1)
    grid.fit(train_texts, train_labels)
    results[f'RF_{vname}'] = grid

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Fitting 3 folds for each of 48 candidates, totalling 144 fits
Fitting 3 folds for each of 48 candidates, totalling 144 fits
Fitting 3 folds for each of 48 candidates, totalling 144 fits


In [9]:
## --- Logistic Regression ---
for vname, vectorizer in vectorizers:
    pipe = Pipeline([
        ('vectorizer', vectorizer),
        ('clf', LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000))
    ])
    param_grid = {
        'vectorizer__max_features': [10000, 20000],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'vectorizer__min_df': [3, 5],
        'clf__C': [0.01, 0.1, 1, 10],
    }
    grid = GridSearchCV(pipe, param_grid, cv=3, scoring='f1', verbose=2, n_jobs=-1)
    grid.fit(train_texts, train_labels)
    results[f'LR_{vname}'] = grid


Fitting 3 folds for each of 32 candidates, totalling 96 fits
Fitting 3 folds for each of 32 candidates, totalling 96 fits
Fitting 3 folds for each of 32 candidates, totalling 96 fits
Fitting 3 folds for each of 32 candidates, totalling 96 fits


In [10]:
## --- Multinomial Naive Bayes ---
for vname, vectorizer in vectorizers:
    pipe = Pipeline([
        ('vectorizer', vectorizer),
        ('clf', MultinomialNB())
    ])
    param_grid = {
        'vectorizer__max_features': [10000, 20000],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'vectorizer__min_df': [3, 5],
        'clf__alpha': [0.1, 1.0, 10.0],
    }
    grid = GridSearchCV(pipe, param_grid, cv=3, scoring='f1', verbose=2, n_jobs=-1)
    grid.fit(train_texts, train_labels)
    results[f'NB_{vname}'] = grid


Fitting 3 folds for each of 24 candidates, totalling 72 fits
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Fitting 3 folds for each of 24 candidates, totalling 72 fits


In [11]:
# === 4. EVALUATE ON VALIDATION SET ===

for name, grid in results.items():
    y_pred = grid.predict(val_texts)
    clf = grid.best_estimator_.named_steps['clf']
    # Get scores for roc_auc
    if hasattr(clf, "predict_proba"):
        y_score = grid.predict_proba(val_texts)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_score = grid.decision_function(val_texts)
    else:
        y_score = None  # fallback

    if y_score is not None:
        try:
            rocauc = roc_auc_score(val_labels, y_score)
            rocauc_str = f"ROC AUC: {rocauc:.4f}"
        except Exception as e:
            rocauc_str = f"ROC AUC: ERROR ({str(e)})"
    else:
        rocauc_str = "ROC AUC: N/A"

    print(f"\n=== Results for {name} ===")
    print("Best Params:", grid.best_params_)
    print("Best CV F1:", grid.best_score_)
    print("Accuracy:", accuracy_score(val_labels, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(val_labels, y_pred))
    print("Classification Report:\n", classification_report(val_labels, y_pred, digits=4))
    print(rocauc_str)
    print("Best Vectorizer:", type(grid.best_estimator_.named_steps['vectorizer']))
    print("Best Classifier:", type(grid.best_estimator_.named_steps['clf']))


=== Results for SVM_tfidf ===
Best Params: {'clf__C': 10, 'vectorizer__max_features': 20000, 'vectorizer__min_df': 3, 'vectorizer__ngram_range': (1, 2)}
Best CV F1: 0.814737537690053
Accuracy: 0.7578692493946732
Confusion Matrix:
 [[113 112]
 [ 88 513]]
Classification Report:
               precision    recall  f1-score   support

           0     0.5622    0.5022    0.5305       225
           1     0.8208    0.8536    0.8369       601

    accuracy                         0.7579       826
   macro avg     0.6915    0.6779    0.6837       826
weighted avg     0.7504    0.7579    0.7534       826

ROC AUC: 0.7609
Best Vectorizer: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
Best Classifier: <class 'sklearn.svm._classes.LinearSVC'>

=== Results for SVM_count ===
Best Params: {'clf__C': 0.01, 'vectorizer__max_features': 20000, 'vectorizer__min_df': 5, 'vectorizer__ngram_range': (1, 2)}
Best CV F1: 0.7966901651432315
Accuracy: 0.7288135593220338
Confusion Matrix:
 [[110 115]

In [12]:
summary = []

for name, grid in results.items():
    y_pred = grid.predict(val_texts)
    clf = grid.best_estimator_.named_steps['clf']
    # Get scores for roc_auc
    if hasattr(clf, "predict_proba"):
        y_score = grid.predict_proba(val_texts)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_score = grid.decision_function(val_texts)
    else:
        y_score = None

    if y_score is not None:
        try:
            rocauc = roc_auc_score(val_labels, y_score)
        except Exception as e:
            rocauc = None
    else:
        rocauc = None

    acc = accuracy_score(val_labels, y_pred)
    f1 = grid.best_score_  # Best CV F1 if scoring='f1', else use f1_score(val_labels, y_pred)
    f1_val = f1 if grid.scoring == 'f1' else classification_report(val_labels, y_pred, output_dict=True)['weighted avg']['f1-score']

    summary.append({
        'model': name,
        'accuracy': acc,
        'f1_val': f1_val,
        'roc_auc': rocauc,
        'best_params': grid.best_params_
    })

# Convert to DataFrame
summary_df = pd.DataFrame(summary)

# For each model family (SVM, RF, LR, NB), find the row with the highest F1 on validation set
summary_df['type'] = summary_df['model'].str.extract(r'^(SVM|RF|LR|NB)')

best_by_type = summary_df.loc[summary_df.groupby('type')['f1_val'].idxmax()]

print("\n=== Best model for each family by validation F1 ===")
print(best_by_type[['type', 'model', 'accuracy', 'f1_val', 'roc_auc', 'best_params']])


=== Best model for each family by validation F1 ===
   type        model  accuracy    f1_val   roc_auc  \
8    LR     LR_tfidf  0.756659  0.808281  0.786977   
12   NB     NB_tfidf  0.773608  0.846684  0.723165   
7    RF  RF_count_sw  0.776029  0.857052  0.783413   
0   SVM    SVM_tfidf  0.757869  0.814738  0.760917   

                                          best_params  
8   {'clf__C': 10, 'vectorizer__max_features': 200...  
12  {'clf__alpha': 0.1, 'vectorizer__max_features'...  
7   {'clf__max_depth': 20, 'clf__n_estimators': 30...  
0   {'clf__C': 10, 'vectorizer__max_features': 200...  


In [21]:
best_by_type.model.to_list()

['LR_tfidf', 'NB_tfidf', 'RF_count_sw', 'SVM_tfidf']

In [13]:
model_dir = '/content/drive/My Drive/EHR_PROJ/MODELS'
os.makedirs(model_dir, exist_ok=True)

for name, grid in results.items():
    model_filename = os.path.join(model_dir, f"{name}_best_model_f1_lowercase.joblib")
    joblib.dump(grid.best_estimator_, model_filename)
    print(f"Saved best model for {name} to {model_filename}")

Saved best model for SVM_tfidf to /content/drive/My Drive/EHR_PROJ/MODELS/SVM_tfidf_best_model_f1_lowercase.joblib
Saved best model for SVM_count to /content/drive/My Drive/EHR_PROJ/MODELS/SVM_count_best_model_f1_lowercase.joblib
Saved best model for SVM_tfidf_sw to /content/drive/My Drive/EHR_PROJ/MODELS/SVM_tfidf_sw_best_model_f1_lowercase.joblib
Saved best model for SVM_count_sw to /content/drive/My Drive/EHR_PROJ/MODELS/SVM_count_sw_best_model_f1_lowercase.joblib
Saved best model for RF_tfidf to /content/drive/My Drive/EHR_PROJ/MODELS/RF_tfidf_best_model_f1_lowercase.joblib
Saved best model for RF_count to /content/drive/My Drive/EHR_PROJ/MODELS/RF_count_best_model_f1_lowercase.joblib
Saved best model for RF_tfidf_sw to /content/drive/My Drive/EHR_PROJ/MODELS/RF_tfidf_sw_best_model_f1_lowercase.joblib
Saved best model for RF_count_sw to /content/drive/My Drive/EHR_PROJ/MODELS/RF_count_sw_best_model_f1_lowercase.joblib
Saved best model for LR_tfidf to /content/drive/My Drive/EHR_PRO

In [14]:
# Set the path where models are saved
model_dir = '/content/drive/My Drive/EHR_PROJ/MODELS'


model_files = [
    "SVM_tfidf_best_model_f1_lowercase.joblib",
    "SVM_count_best_model_f1_lowercase.joblib",
    "SVM_tfidf_sw_best_model_f1_lowercase.joblib",
    "SVM_count_sw_best_model_f1_lowercase.joblib",
    "RF_tfidf_best_model_f1_lowercase.joblib",
    "RF_count_best_model_f1_lowercase.joblib",
    "RF_tfidf_sw_best_model_f1_lowercase.joblib",
    "RF_count_sw_best_model_f1_lowercase.joblib",
    "LR_tfidf_best_model_f1_lowercase.joblib",
    "LR_count_best_model_f1_lowercase.joblib",
    "LR_tfidf_sw_best_model_f1_lowercase.joblib",
    "LR_count_sw_best_model_f1_lowercase.joblib",
    "NB_tfidf_best_model_f1_lowercase.joblib",
    "NB_count_best_model_f1_lowercase.joblib",
    "NB_tfidf_sw_best_model_f1_lowercase.joblib",
    "NB_count_sw_best_model_f1_lowercase.joblib",
]


results = {}
for file in model_files:
    model_name = file.replace("_best_model_f1_lowercase.joblib", "")
    model_path = os.path.join(model_dir, file)  # join path correctly
    if not os.path.exists(model_path):
        print(f"File not found: {model_path}")
        continue
    model = joblib.load(model_path)
    preds = model.predict(val_texts)  # test_texts should be your test data list/array
    results[model_name] = preds

# Combine all predictions into a DataFrame
preds_df = pd.DataFrame(results)

In [15]:
preds_df

,SVM_tfidf,SVM_count,SVM_tfidf_sw,SVM_count_sw,RF_tfidf,RF_count,RF_tfidf_sw,RF_count_sw,LR_tfidf,LR_count,LR_tfidf_sw,LR_count_sw,NB_tfidf,NB_count,NB_tfidf_sw,NB_count_sw
0,1,1,1,0,1,1,1,1,1,1,0,0,1,1,1,1
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,1
3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
4,0,0,0,1,1,1,1,1,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
822,0,0,0,0,1,1,1,1,0,0,0,0,1,1,1,1
823,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
824,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [16]:
# prompt: add true labels to the preds_df

preds_df['true_label'] = val_labels
preds_df

,SVM_tfidf,SVM_count,SVM_tfidf_sw,SVM_count_sw,RF_tfidf,RF_count,RF_tfidf_sw,RF_count_sw,LR_tfidf,LR_count,LR_tfidf_sw,LR_count_sw,NB_tfidf,NB_count,NB_tfidf_sw,NB_count_sw,true_label
0,1,1,1,0,1,1,1,1,1,1,0,0,1,1,1,1,1
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,0,0,0,0,0,1,0,1,0,0,0,0,1,1,1,1,0
3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0
4,0,0,0,1,1,1,1,1,0,0,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
822,0,0,0,0,1,1,1,1,0,0,0,0,1,1,1,1,1
823,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
824,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
# prompt: Compute accuracy for each column

# Calculate accuracy for each model and store in a dictionary
accuracies = {}
for col in preds_df.columns:
    if col != 'true_label':
        accuracies[col] = accuracy_score(preds_df['true_label'], preds_df[col])

# Print the accuracy for each column
print("\nAccuracy for each model on the validation set:")
for model_name, acc in accuracies.items():
    print(f"{model_name}: {acc:.4f}")


Accuracy for each model on the validation set:
SVM_tfidf: 0.7579
SVM_count: 0.7288
SVM_tfidf_sw: 0.7554
SVM_count_sw: 0.7107
RF_tfidf: 0.7772
RF_count: 0.7676
RF_tfidf_sw: 0.7797
RF_count_sw: 0.7760
LR_tfidf: 0.7567
LR_count: 0.7264
LR_tfidf_sw: 0.7530
LR_count_sw: 0.7228
NB_tfidf: 0.7736
NB_count: 0.7676
NB_tfidf_sw: 0.7591
NB_count_sw: 0.7651


In [ ]:
# prompt: For the four best models ['LR_tfidf', 'NB_tfidf', 'RF_count_sw', 'SVM_tfidf'], compute all metrics. Use loaded models

best_models_to_evaluate = ['LR_tfidf', 'NB_tfidf', 'RF_count_sw', 'SVM_tfidf']

print("\n=== Metrics for the four best models ===")

metrics_summary = {}

for model_name in best_models_to_evaluate:
    if model_name in results:
        model = joblib.load(os.path.join(model_dir, f"{model_name}_best_model_f1_lowercase.joblib")) # Load the specific best model
        print(f"\n--- Evaluating {model_name} ---")

        y_pred = model.predict(val_texts)

        # Get scores for roc_auc
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(val_texts)[:, 1]
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(val_texts)
        else:
            y_score = None

        acc = accuracy_score(val_labels, y_pred)
        cm = confusion_matrix(val_labels, y_pred)
        report = classification_report(val_labels, y_pred, digits=4)

        rocauc = None
        if y_score is not None:
            try:
                rocauc = roc_auc_score(val_labels, y_score)
                rocauc_str = f"ROC AUC: {rocauc:.4f}"
            except Exception as e:
                rocauc_str = f"ROC AUC: ERROR ({str(e)})"
        else:
            rocauc_str = "ROC AUC: N/A"

        print("Accuracy:", acc)
        print("Confusion Matrix:\n", cm)
        print("Classification Report:\n", report)
        print(rocauc_str)

        # Store metrics in the summary dictionary
        metrics_summary[model_name] = {
            'accuracy': acc,
            'confusion_matrix': cm.tolist(), # Convert numpy array to list for storage
            'classification_report': report,
            'roc_auc': rocauc
        }
    else:
        print(f"\n--- Model {model_name} not found ---")

# Optional: Print the summary dictionary
# print("\n=== Summary of Metrics ===")
# for model_name, metrics in metrics_summary.items():
#     print(f"\nModel: {model_name}")
#     print(f"  Accuracy: {metrics['accuracy']:.4f}")
#     print(f"  ROC AUC: {metrics['roc_auc']:.4f}" if metrics['roc_auc'] is not None else "  ROC AUC: N/A")
#     print(f"  Classification Report:\n{metrics['classification_report']}")


In [23]:
best_models_to_evaluate = ['LR_tfidf', 'NB_tfidf', 'RF_count_sw', 'SVM_tfidf']

print("\n=== Metrics for the four best models ===")

metrics_summary = {}

for model_name in best_models_to_evaluate:
    if model_name in results:
        model = joblib.load(os.path.join(model_dir, f"{model_name}_best_model_f1_lowercase.joblib")) # Load the specific best model
        print(f"\n--- Evaluating {model_name} ---")

        y_pred = model.predict(val_texts)

        # Get scores for roc_auc
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(val_texts)[:, 1]
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(val_texts)
        else:
            y_score = None

        acc = accuracy_score(val_labels, y_pred)
        cm = confusion_matrix(val_labels, y_pred)
        report = classification_report(val_labels, y_pred, digits=4)

        rocauc = None
        if y_score is not None:
            try:
                rocauc = roc_auc_score(val_labels, y_score)
                rocauc_str = f"ROC AUC: {rocauc:.4f}"
            except Exception as e:
                rocauc_str = f"ROC AUC: ERROR ({str(e)})"
        else:
            rocauc_str = "ROC AUC: N/A"

        print("Accuracy:", acc)
        print("Confusion Matrix:\n", cm)
        print("Classification Report:\n", report)
        print(rocauc_str)

        # Store metrics in the summary dictionary
        metrics_summary[model_name] = {
            'accuracy': acc,
            'confusion_matrix': cm.tolist(), # Convert numpy array to list for storage
            'classification_report': report,
            'roc_auc': rocauc
        }
    else:
        print(f"\n--- Model {model_name} not found ---")


=== Metrics for the four best models ===

--- Evaluating LR_tfidf ---
Accuracy: 0.7566585956416465
Confusion Matrix:
 [[135  90]
 [111 490]]
Classification Report:
               precision    recall  f1-score   support

           0     0.5488    0.6000    0.5732       225
           1     0.8448    0.8153    0.8298       601

    accuracy                         0.7567       826
   macro avg     0.6968    0.7077    0.7015       826
weighted avg     0.7642    0.7567    0.7599       826

ROC AUC: 0.7870

--- Evaluating NB_tfidf ---
Accuracy: 0.7736077481840193
Confusion Matrix:
 [[ 67 158]
 [ 29 572]]
Classification Report:
               precision    recall  f1-score   support

           0     0.6979    0.2978    0.4174       225
           1     0.7836    0.9517    0.8595       601

    accuracy                         0.7736       826
   macro avg     0.7407    0.6248    0.6385       826
weighted avg     0.7602    0.7736    0.7391       826

ROC AUC: 0.7232

--- Evaluating RF_count

In [18]:
# Optionally, save the predictions DataFrame to a CSV file
preds_df.to_csv('/content/drive/My Drive/EHR_PROJ/Results/predictions_traditional_MLs.csv', index=False)
print("\nPredictions DataFrame saved to /content/drive/My Drive/EHR_PROJ/Results/predictions_traditional_MLs.csv")


Predictions DataFrame saved to /content/drive/My Drive/EHR_PROJ/Results/predictions_traditional_MLs.csv
